<a href="https://colab.research.google.com/github/MohammadJoenathan/TextMining_PenerapanTextVectorizer/blob/main/2218060MohammadJoenathantrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import semua modul Library yang dibutuhkan

In [1]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 8.3 MB/s eta 0:00:00


In [2]:
!pip install Sastrawi scikit-learn joblib

In [3]:
import json
import joblib
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# 2. Menghubungkan Google Colab dengan Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Training TF-IDF dan Membuat Vector Database

In [5]:
# Load stemmer bahasa indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Fungsi preprocessing text untuk membersihkan teks sebelum dibuat vektor tf-idf
def preprocess_text(text):
    text = text.lower()  # ubah huruf menjadi kecil semua
    text = re.sub(r'[^\w\s]', '', text)  # hapus tanda baca/simbol
    words = text.split()  # pecah kalimat menjadi kata-kata
    stemmed_words = [stemmer.stem(word) for word in words]  # stemming kata
    return " ".join(stemmed_words)  # gabungkan kembali menjadi kalimat

# Load dataset dari data.json
with open("/content/drive/MyDrive/Colab Notebooks/TextMiningUTS/data.json", "r", encoding="utf-8") as file:
    corpus = json.load(file)["qa_corpus"]  # ambil list question-answer dari qa_corpus

# Ambil question dan answer dari dataset
questions = [item["question"] for item in corpus]
answers = [item["answer"] for item in corpus]

# Preprocessing question dan answer
preprocessed_questions = [preprocess_text(q) for q in questions]
preprocessed_answers = [preprocess_text(a) for a in answers]

# Gabungkan corpus untuk training tf-idf
combined_corpus = preprocessed_questions + preprocessed_answers

# Training tf-idf vectorizer
vectorizer = TfidfVectorizer()
vectorizer.fit(combined_corpus)

# Simpan model vectorizer ke google drive
joblib.dump(vectorizer, "/content/drive/MyDrive/Colab Notebooks/TextMiningUTS/vectorizer.joblib")
print("vectorizer berhasil disimpan ke google drive (vectorizer.joblib)")

# Ubah pertanyaan menjadi vektor tf-idf
question_vectors = vectorizer.transform(preprocessed_questions)

# Simpan vector database ke vector.json
vector_data = []
for question, answer, vector in zip(questions, answers, question_vectors):
    vector_data.append({
        "question": question,
        "answer": answer,
        "vector": vector.toarray().tolist()[0]  # ubah vector menjadi list agar bisa disimpan di json
    })

# Simpan vector.json ke google drive
with open("/content/drive/MyDrive/Colab Notebooks/TextMiningUTS/vector.json", "w", encoding="utf-8") as file:
    json.dump(vector_data, file, indent=4, ensure_ascii=False)

print("vector database berhasil disimpan ke google drive (vector.json)")
print("training selesai!")

vectorizer berhasil disimpan ke google drive (vectorizer.joblib)
vector database berhasil disimpan ke google drive (vector.json)
training selesai!
